# 08. Reihenfolgen: Wer muss zuerst dran sein?

Manchmal sind die Dinge nicht gleichberechtigt: Die Socken müssen vor den Schuhen an, ein Kurs setzt einen anderen voraus, eine Aufgabe in einem
Projekt kann erst beginnen, wenn eine andere fertig ist. Gesucht ist dann eine **Reihenfolge**, in der **alle Vorgaben „A vor B“** erfüllt sind. Das Verfahren
dazu heißt **topologische Sortierung**. In [Notebook 05](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb) hast du einen Graphen als Wörterbuch mit Nachbarlisten gespeichert.
Hier ist es genauso, nur zeigen die Kanten in **eine** Richtung: von der Sache, die zuerst dran sein muss, zu der, die danach kommt.

Verwandt ist das mit [Notebook 07](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb): Dort bestimmte die kleinste Entfernung, welcher Knoten als Nächstes bearbeitet wird. Hier bestimmen
die **offenen Vorgänger**, wer als Nächstes dran ist.

In diesem Notebook lernst du:

- Vorgaben „A vor B“ als **gerichteten Graphen** mit Nachfolgerlisten zu speichern,
- den **Eingangsgrad** (Zahl der noch offenen Vorgänger) zu zählen und danach zu sortieren (Verfahren von **Kahn**),
- zu erkennen, wann es **keine** Reihenfolge gibt: Die Vorgaben bilden einen **Kreis** (ein Widerspruch), und man kann einen konkreten Kreis ausgeben,
- dieselbe Sortierung rekursiv mit **Tiefensuche** zu schreiben,
- zu **zählen**, wie viele gültige Reihenfolgen es gibt (Backtracking wie in Notebook 02), und in **Schichten** zu sortieren (was kann gleichzeitig geschehen?),
- die Laufzeit **O(Knoten + Kanten)** zu begründen.

In der Wettbewerbs-Ecke ordnest du Klausuraufgaben nach Schwierigkeit und meldest Widersprüche.

## 1. Vorgaben als Graph

Jemand zieht sich an. Folgende Vorgaben gelten (jeweils „links muss vor rechts kommen“):

| Vorgabe | Grund |
|---|---|
| Unterhose vor Hose | |
| Hose vor Schuhe | |
| Socken vor Schuhe | |
| Hose vor Gürtel | |
| Hemd vor Pullover | |
| Pullover vor Jacke | |
| Gürtel vor Jacke | |

Jedes Kleidungsstück ist ein **Knoten**, jede Vorgabe „A vor B“ eine **Kante** `A → B`. Weil der Pfeil eine Richtung hat, heißt der Graph **gerichtet**. Wir speichern ihn wie in
Notebook 05 als Wörterbuch: Zu jedem Knoten gehört die Liste seiner **Nachfolger**, also derer, die direkt nach ihm kommen müssen. Hat ein Knoten keine Nachfolger, steht eine leere Liste dabei.

#### Frage 1.1

Zeichne die sieben Vorgaben als Pfeile auf Papier (acht Kleidungsstücke).

1. Welche Kleidungsstücke kann man **als Erstes** anziehen, welche **nie als Erstes**?
2. Schreibe eine vollständige Reihenfolge auf, die alle Vorgaben erfüllt.
3. Gibt es nur eine solche Reihenfolge?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Als Erstes können alle Stücke, vor denen nichts anderes kommen muss: **Unterhose**, **Socken** und **Hemd**. Nie als Erstes kommen Hose, Schuhe, Gürtel, Pullover und Jacke, denn bei jedem von ihnen
   zeigt ein Pfeil darauf (Vorgänger).
2. Zum Beispiel: Unterhose, Hose, Gürtel, Hemd, Pullover, Jacke, Socken, Schuhe. Prüfe jede Vorgabe: Die Unterhose steht vor der Hose, die Hose vor den Schuhen, die Socken vor den Schuhen, die Hose vor dem Gürtel, das Hemd vor
   dem Pullover, der Pullover vor der Jacke und der Gürtel vor der Jacke. Alles erfüllt.
3. Nein, es gibt **viele**: Man kann zum Beispiel Socken auch ganz am Anfang anziehen. Wie viele es sind, zählen wir in Abschnitt 6.

</details>

#### Aufgabe 1.2

Schreibe `baue_graph(knoten, paare)`: Sie bekommt die Liste aller Knoten und die Liste der Paare `(a, b)` („a vor b“) und gibt das Wörterbuch zurück, in dem zu jedem Knoten die Liste seiner Nachfolger steht
(auch Knoten ohne Nachfolger kommen vor, mit leerer Liste).

**Erwartete Ausgabe:**
```
Hemd -> ['Pullover']
Hose -> ['Schuhe', 'Gürtel']
Gürtel -> ['Jacke']
Jacke -> []
Pullover -> ['Jacke']
Schuhe -> []
Socken -> ['Schuhe']
Unterhose -> ['Hose']
```

In [ ]:
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    # dein Code
    return graph

anziehen = baue_graph(kleidung, vorgaben)
for k in anziehen:
    print(k, '->', anziehen[k])

<details>
<summary>Tipp anzeigen</summary>

Zuerst für jeden Knoten `graph[k] = []`. Danach für jedes Paar `(a, b)`: `graph[a].append(b)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

anziehen = baue_graph(kleidung, vorgaben)
for k in anziehen:
    print(k, '->', anziehen[k])
```

Das Hemd hat den Nachfolger Pullover, die Hose hat zwei Nachfolger (Schuhe, Gürtel), die Jacke und die Schuhe haben keine. Dass auch Knoten ohne Nachfolger im Wörterbuch stehen, ist wichtig: Sonst
würde `graph[k]` bei der Jacke einen Fehler melden.

</details>

## 2. Eingangsgrad und das Verfahren von Kahn

Wer darf als Nächstes drankommen? Genau die, bei denen **kein offener Vorgänger** mehr wartet. Die Zahl der Vorgänger eines Knotens heißt sein **Eingangsgrad**: Es ist die Zahl der Pfeile, die auf ihn zeigen.

Das Verfahren (nach Arthur Kahn):

1. Berechne für jeden Knoten den Eingangsgrad.
2. Alle Knoten mit Eingangsgrad 0 sind **bereit**.
3. Solange ein Knoten bereit ist: Nimm einen, hänge ihn **hinten an die Reihenfolge an**, und senke bei allen seinen Nachfolgern den Eingangsgrad um 1 (ein Vorgänger weniger wartet). Kommt ein Nachfolger dabei auf 0, wird er bereit.

Unter den bereiten Knoten ist **jeder** erlaubt. Damit das Ergebnis jedes Mal gleich ist, nehmen wir den **kleinsten** nach dem Alphabet (`min`).

#### Aufgabe 2.1

Schreibe `eingangsgrade(graph)`: Sie gibt ein Wörterbuch zurück, in dem zu jedem Knoten steht, auf wie viele Pfeile er als Ziel zeigt.

**Erwartete Ausgabe:**
```
Gürtel 1
Hemd 0
Hose 1
Jacke 2
Pullover 1
Schuhe 2
Socken 0
Unterhose 0
```

In [ ]:
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

anziehen = baue_graph(kleidung, vorgaben)

def eingangsgrade(graph):
    eingang = {}
    # dein Code
    return eingang

eingang = eingangsgrade(anziehen)
for k in sorted(eingang):
    print(k, eingang[k])

<details>
<summary>Tipp anzeigen</summary>

Setze zuerst jeden Eingangsgrad auf 0. Gehe dann alle Knoten `k` und alle Nachfolger `n` in `graph[k]` durch und erhöhe `eingang[n]` um 1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

anziehen = baue_graph(kleidung, vorgaben)

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

eingang = eingangsgrade(anziehen)
for k in sorted(eingang):
    print(k, eingang[k])
```

Die **Schuhe** haben den Eingangsgrad 2 (Hose und Socken), die **Jacke** auch (Pullover und Gürtel). Hose, Gürtel und Pullover haben je einen Vorgänger.
Hemd, Socken und Unterhose haben den Eingangsgrad 0: Das sind die Knoten, mit denen man anfangen kann, wie in Frage 1.1.

</details>

#### Aufgabe 2.2

Schreibe `sortiere(graph)` nach dem Verfahren von Kahn. Die Funktion gibt die Reihenfolge als Liste zurück. Wähle unter den bereiten Knoten immer den alphabetisch kleinsten (`min(bereit)`, danach mit `bereit.remove(...)` entfernen).

**Erwartete Ausgabe:**
```
['Hemd', 'Pullover', 'Socken', 'Unterhose', 'Hose', 'Gürtel', 'Jacke', 'Schuhe']
```

In [ ]:
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

anziehen = baue_graph(kleidung, vorgaben)

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    reihenfolge = []
    # dein Code
    return reihenfolge

print(sortiere(anziehen))

<details>
<summary>Tipp anzeigen</summary>

Gerüst: Erst alle Knoten mit Eingangsgrad 0 in `bereit`. Dann `while len(bereit) > 0:` den kleinsten Knoten nehmen, entfernen, an `reihenfolge` hängen, bei allen Nachfolgern `eingang[n]` um 1 senken, und wenn der Wert 0 wird, den Nachfolger an `bereit` hängen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

anziehen = baue_graph(kleidung, vorgaben)

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

print(sortiere(anziehen))
```

Bereit sind zuerst Hemd, Socken und Unterhose. Das Hemd ist alphabetisch das kleinste, danach wird der Pullover bereit (sein einziger Vorgänger ist fertig). Von Pullover, Socken und Unterhose kommt der **Pullover** dran, dann die Socken, dann die Unterhose.
Erst nach der Unterhose ist die Hose bereit, erst nach der Hose und den Socken die Schuhe, erst nach Gürtel und Pullover die Jacke. In dieser Reihenfolge stehen alle Vorgaben, Probe in Aufgabe 2.3.
Der Unterschied zu Notebook 07: Dort entschied die **Entfernung** über den nächsten Knoten, hier nur, ob **alle Vorgänger fertig** sind.

</details>

#### Aufgabe 2.3

Schreibe `ist_gueltig(graph, reihenfolge)`: Sie gibt `True` zurück, wenn jeder Knoten des Graphen genau einmal in `reihenfolge` vorkommt **und** für jede Kante `k → n` der Knoten `k` weiter vorn steht als `n`. Merke dir dazu zu jedem Knoten seine Stelle in einem Wörterbuch.
Im Aufruf unten ist die zweite Reihenfolge falsch (Schuhe zuerst), die dritte unvollständig (die Schuhe fehlen).

**Erwartete Ausgabe:**
```
True
False
False
```

In [ ]:
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

anziehen = baue_graph(kleidung, vorgaben)

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    # dein Code
    return False

print(ist_gueltig(anziehen, sortiere(anziehen)))
print(ist_gueltig(anziehen, ['Schuhe', 'Socken', 'Unterhose', 'Hose', 'Hemd', 'Pullover', 'Gürtel', 'Jacke']))
print(ist_gueltig(anziehen, ['Hemd', 'Pullover', 'Socken', 'Unterhose', 'Hose', 'Gürtel', 'Jacke']))

<details>
<summary>Tipp anzeigen</summary>

Zuerst `sorted(reihenfolge) != sorted(graph)` (die Schlüssel des Wörterbuchs) prüfen. Dann `stelle[reihenfolge[i]] = i` füllen und für jede Kante `k → n` den Wert `stelle[k]` mit `stelle[n]` vergleichen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

anziehen = baue_graph(kleidung, vorgaben)

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

print(ist_gueltig(anziehen, sortiere(anziehen)))
print(ist_gueltig(anziehen, ['Schuhe', 'Socken', 'Unterhose', 'Hose', 'Hemd', 'Pullover', 'Gürtel', 'Jacke']))
print(ist_gueltig(anziehen, ['Hemd', 'Pullover', 'Socken', 'Unterhose', 'Hose', 'Gürtel', 'Jacke']))
```

Eine solche Probe ist bei Wettbewerbsaufgaben sehr nützlich: Sie ist einfach, unabhängig vom Verfahren und fängt Fehler ab. Beim dritten Aufruf fehlt ein Knoten, deshalb `False`.

</details>

#### Frage 2.4

Anziehen hat 8 Knoten und 7 Kanten.

1. Wie oft wird in `eingangsgrade` eine Kante angesehen? Wie oft in `sortiere` insgesamt (Schleife `for n in graph[k]`)? Wie oft wird ein Knoten aus `bereit` genommen?
2. Wie schätzt du daraus die Laufzeit für einen Graphen mit `K` Knoten und `E` Kanten ab (O-Notation)?
3. Warum ist `min(bereit)` hier kein Problem, obwohl `min` die ganze Liste ansieht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. In `eingangsgrade` wird **jede** Kante genau einmal angesehen (7-mal), in `sortiere` ebenfalls (jeder Knoten wird genau einmal herausgenommen, dabei werden alle seine ausgehenden Kanten angesehen: zusammen 7). Aus `bereit` wird
   jeder Knoten genau **einmal** genommen (8-mal).
2. Jeder Knoten wird einmal bearbeitet, jede Kante einmal angesehen: **O(K + E)**. Das ist **linear**, die Sortierung ist also sehr schnell. Zum Vergleich: Die Reihenfolgen durchzuprobieren würde bei 8 Knoten schon bis zu 8! = 40 320
   Anordnungen bedeuten.
3. `bereit` ist meist sehr kurz (hier höchstens 3 Knoten). Bei großen Graphen kann `bereit` aber groß werden, dann kostet jedes `min` viel; dann nimmt man statt `min` einfach `bereit.pop()` (oder `pop(0)` wie in Notebook 05) und
   verzichtet auf die alphabetische Reihenfolge, oder man verwendet `heapq` (kommt in einem späteren Notebook). Bei den Wettbewerbsaufgaben reicht `min`, weil nur einige Dutzend Knoten vorkommen.

</details>

## 3. Wenn es keine Reihenfolge gibt: Kreise

Ein Studienplan mit Voraussetzungen („X vor Y“) für sechs Kurse:

- Programmieren vor Algorithmen
- Programmieren vor Datenbanken
- Algorithmen vor Projekt
- Datenbanken vor Projekt
- Netze vor Projekt
- Projekt vor Abschluss
- **Projekt vor Datenbanken** (ein Fehler bei der Eingabe?)

Die letzte Vorgabe widerspricht den anderen: Die Datenbanken müssen vor dem Projekt kommen **und** das Projekt vor den Datenbanken. Folgt man den Pfeilen `Datenbanken → Projekt → Datenbanken`, kommt man zum Ausgangspunkt zurück: ein **Kreis**.
Ein Graph **ohne** Kreis heißt **azyklisch**. Nur dann gibt es eine Reihenfolge.

Was passiert bei Kahn? Die Knoten im Kreis haben immer einen Vorgänger, der selbst erst bearbeitet werden müsste. Ihr Eingangsgrad kommt nie auf 0, und alles, was von ihnen abhängt, bleibt ebenfalls liegen.
Dann ist die zurückgegebene Liste **kürzer** als die Zahl der Knoten.

#### Aufgabe 3.1

Schreibe zwei Funktionen auf der Grundlage von `sortiere`:

- `uebrige_knoten(graph)`: die Knoten, die in der Sortierung **nicht** vorkommen (in der Reihenfolge des Wörterbuchs),
- `hat_reihenfolge(graph)`: `True`, wenn alle Knoten sortiert wurden.

Der Aufruf unten vergleicht den Studienplan ohne und mit der falschen Vorgabe.

**Erwartete Ausgabe:**
```
6 von 6 Kursen: ['Netze', 'Programmieren', 'Algorithmen', 'Datenbanken', 'Projekt', 'Abschluss']
Reihenfolge möglich: True
übrig: []
3 von 6 Kursen: ['Netze', 'Programmieren', 'Algorithmen']
Reihenfolge möglich: False
übrig: ['Datenbanken', 'Projekt', 'Abschluss']
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

def uebrige_knoten(graph):
    # dein Code
    return []


def hat_reihenfolge(graph):
    # dein Code
    return False

for plan in [studienplan, fehlerhaft]:
    reihenfolge = sortiere(plan)
    print(len(reihenfolge), 'von', len(plan), 'Kursen:', reihenfolge)
    print('Reihenfolge möglich:', hat_reihenfolge(plan))
    print('übrig:', uebrige_knoten(plan))

<details>
<summary>Tipp anzeigen</summary>

`fertig = sortiere(graph)`. Bei `uebrige_knoten` alle `k` in `graph` nehmen, die nicht in `fertig` stehen. Bei `hat_reihenfolge` die Länge von `fertig` mit der Zahl der Knoten vergleichen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

for plan in [studienplan, fehlerhaft]:
    reihenfolge = sortiere(plan)
    print(len(reihenfolge), 'von', len(plan), 'Kursen:', reihenfolge)
    print('Reihenfolge möglich:', hat_reihenfolge(plan))
    print('übrig:', uebrige_knoten(plan))
```

Ohne die falsche Vorgabe werden alle 6 Kurse sortiert. Mit der falschen Vorgabe sind es nur **3**: Programmieren, Algorithmen und Netze. Übrig bleiben Datenbanken, Projekt und Abschluss. Der Abschluss steckt nicht selbst im Kreis, er hängt
nur vom Projekt ab und kann deshalb nicht drankommen.

</details>

### Einen Kreis ausgeben

„Es gibt einen Widerspruch“ hilft wenig, wenn man ihn finden will. Besser ist ein **konkreter Kreis**, den man dem Benutzer zeigen kann. Ein Trick: Jeder übrig gebliebene Knoten hat einen Vorgänger, der **auch übrig** ist (sonst wäre sein
Eingangsgrad auf 0 gekommen). Man geht also von einem übrigen Knoten **rückwärts** von Vorgänger zu Vorgänger. Weil es nur endlich viele Knoten gibt, kommt man irgendwann zu einem Knoten, den man schon besucht hat. Das Stück des Wegs ab diesem Knoten
ist ein Kreis, rückwärts gelesen. Dreht man es um, hat man die Pfeile in der richtigen Richtung.

#### Aufgabe 3.2

Die Funktion `finde_kreis(graph)` ist unten vorgegeben (lies sie in Ruhe: `weg` ist der Weg rückwärts, `vorgaenger[n]` ist ein übriger Knoten mit Pfeil nach `n`). Sie braucht
`uebrige_knoten` aus der Aufgabe 3.1. Führe die Zelle aus und beantworte danach die Frage 3.3.

**Erwartete Ausgabe:**
```
[]
['Projekt', 'Datenbanken']
Projekt -> Datenbanken -> Projekt
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

print(finde_kreis(studienplan))
kreis = finde_kreis(fehlerhaft)
print(kreis)
print(' -> '.join(kreis + [kreis[0]]))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

print(finde_kreis(studienplan))
kreis = finde_kreis(fehlerhaft)
print(kreis)
print(' -> '.join(kreis + [kreis[0]]))
```

Der Kreis ist `Projekt -> Datenbanken -> Projekt`. Der Weg rückwärts beginnt beim Knoten `Datenbanken` (dem ersten übrigen Knoten im Wörterbuch): sein übriger Vorgänger ist `Projekt`, davor wieder `Datenbanken`, einer, den man schon hat.

</details>

#### Frage 3.3

1. Warum hat **jeder** übrig gebliebene Knoten einen Vorgänger, der auch übrig ist? (Denke an den Eingangsgrad.)
2. Der Abschluss ist übrig, liegt aber nicht im Kreis. Wie kann `finde_kreis` trotzdem funktionieren, wenn man beim Abschluss anfängt? Probiere es auf Papier mit `weg = ['Abschluss']`.
3. Ein Student sagt: „Dann streichen wir einfach eine Vorgabe aus dem Kreis.“ Warum kann ein Programm nicht allein entscheiden, welche?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der Eingangsgrad eines übrigen Knotens ist **größer als 0**: Es warten noch Vorgänger. Wären alle seine Vorgänger sortiert worden, hätte sein Eingangsgrad 0 erreicht, und er wäre sortiert worden. Also ist mindestens ein Vorgänger **nicht** sortiert, ist also übrig.
2. Vom Abschluss geht der Weg rückwärts zum Projekt, von dort zu einem übrigen Vorgänger des Projekts (hier `Datenbanken`), dann wieder zum Projekt: Das Projekt kommt zum zweiten Mal vor. `weg[weg.index(davor):]` schneidet
   ab dem ersten Auftreten des Projekts aus. Der Abschluss steht nur am Anfang von `weg` und fällt beim Ausschneiden weg. Er gehört nicht zum Kreis, sondern hängt nur dahinter.
3. Das Programm weiß nicht, welche Vorgabe der **Fehler** ist: Vielleicht ist „Datenbanken vor Projekt“ falsch, vielleicht „Projekt vor Datenbanken“. Das entscheidet ein Mensch. Das Programm kann nur den Kreis **zeigen**.

</details>

## 4. Finde den Fehler

#### Aufgabe 4.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere_falsch(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = bereit.pop(0)
        reihenfolge.append(k)
        for n in graph[k]:
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

schuhe = {'Socken': ['Schuhe'], 'Hose': ['Schuhe', 'Gürtel'], 'Schuhe': [], 'Gürtel': []}
print(sortiere_falsch(schuhe))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
['Socken', 'Hose']
```

Ausgegeben wird `['Socken', 'Hose']`. Die Schuhe und der Gürtel fehlen! Der Fehler: Beim Bearbeiten von `Socken` wird der Eingangsgrad der Nachfolger **nicht gesenkt** (`eingang[n] = eingang[n] - 1` fehlt).
Die Schuhe haben den Eingangsgrad 2 (Socken und Hose) und die Gürtel den Eingangsgrad 1. Sie bleiben dort stehen, die Bedingung `eingang[n] == 0` ist nie wahr.
Nur Knoten, die schon am Anfang den Eingangsgrad 0 hatten, kommen vor. Die Eingangsgrade müssen **mitgeführt** werden: nach jedem bearbeiteten Knoten bei allen Nachfolgern um 1 senken **und dann** auf 0 prüfen.

</details>

#### Aufgabe 4.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def sortiere(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = bereit.pop(0)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

plan = {'A': ['B'], 'B': ['C'], 'C': ['A', 'D'], 'D': [], 'E': ['D']}
reihenfolge = sortiere(plan)
print('Fertig! Reihenfolge:', ' '.join(reihenfolge))
print(len(reihenfolge), 'von', len(plan), 'Knoten')

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
Fertig! Reihenfolge: E
1 von 5 Knoten
```

Ausgegeben wird `Fertig! Reihenfolge: E` und `1 von 5 Knoten`. A, B und C bilden einen Kreis (`A → B → C → A`), keiner von ihnen kommt je auf den Eingangsgrad 0, und D hängt außerdem von C ab. Nur E kann sortiert werden.
Das Programm gibt trotzdem „Fertig!“ aus, weil niemand geprüft hat, ob **alle** Knoten dran waren. Eine Liste, die zu kurz ist, sieht wie eine Antwort aus. Nach `sortiere` muss man immer
`len(reihenfolge) == len(plan)` prüfen, und bei einem Widerspruch statt der Reihenfolge eine Meldung (am besten mit einem Kreis) ausgeben.

</details>

## 5. Mit Tiefensuche (rekursiv)

Es gibt einen zweiten Weg zur Reihenfolge, der ohne Eingangsgrade auskommt. Die Idee: Wenn man einen Knoten ganz **abgearbeitet** hat, also schon alle seine Nachfolger, dann kann er in der Reihenfolge **hinter** all diesen Nachfolgern stehen.
Man arbeitet jeden Knoten rekursiv ab (wie in Notebook 02) und hängt ihn **nach** seinen Nachfolgern an eine Liste. Am Ende dreht man die Liste um.

Damit man Kreise bemerkt, hat jeder Knoten einen von drei **Zuständen** (wie in Notebook 06):

- `0`: noch nicht besucht,
- `1`: **in Arbeit** (steht auf dem aktuellen Weg, seine Nachfolger sind noch nicht fertig),
- `2`: fertig.

Trifft man auf einen Nachfolger im Zustand `1`, ist man im Kreis: Man ist von diesem Knoten über Pfeile zu ihm zurückgekommen.

#### Aufgabe 5.1

Die Funktion `sortiere_dfs(graph)` ist unten vorgegeben: Sie gibt die Reihenfolge zurück oder `None`, wenn es einen Kreis gibt. Führe die Zelle aus und lies den Code Zeile für Zeile mit. Die Hilfsfunktion `besuche` ist innerhalb von
`sortiere_dfs` definiert, damit sie auf `zustand` und `umgekehrt` zugreifen kann. Danach beantworte Frage 5.2.

**Erwartete Ausgabe:**
```
['Hemd', 'Pullover', 'Socken', 'Unterhose', 'Hose', 'Gürtel', 'Jacke', 'Schuhe']
True
['Netze', 'Programmieren', 'Algorithmen', 'Datenbanken', 'Projekt', 'Abschluss']
None
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def sortiere_dfs(graph):
    zustand = {}
    for k in graph:
        zustand[k] = 0
    umgekehrt = []

    def besuche(k):
        zustand[k] = 1                             # in Arbeit
        for n in sorted(graph[k], reverse=True):
            if zustand[n] == 1:
                return False                       # n ist in Arbeit: Kreis
            if zustand[n] == 0:
                if not besuche(n):
                    return False
        zustand[k] = 2                             # fertig
        umgekehrt.append(k)
        return True

    for k in sorted(graph, reverse=True):
        if zustand[k] == 0:
            if not besuche(k):
                return None                        # Kreis gefunden
    umgekehrt.reverse()
    return umgekehrt

print(sortiere_dfs(anziehen))
print(ist_gueltig(anziehen, sortiere_dfs(anziehen)))
print(sortiere_dfs(studienplan))
print(sortiere_dfs(fehlerhaft))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def sortiere_dfs(graph):
    zustand = {}
    for k in graph:
        zustand[k] = 0
    umgekehrt = []

    def besuche(k):
        zustand[k] = 1                             # in Arbeit
        for n in sorted(graph[k], reverse=True):
            if zustand[n] == 1:
                return False                       # n ist in Arbeit: Kreis
            if zustand[n] == 0:
                if not besuche(n):
                    return False
        zustand[k] = 2                             # fertig
        umgekehrt.append(k)
        return True

    for k in sorted(graph, reverse=True):
        if zustand[k] == 0:
            if not besuche(k):
                return None                        # Kreis gefunden
    umgekehrt.reverse()
    return umgekehrt

print(sortiere_dfs(anziehen))
print(ist_gueltig(anziehen, sortiere_dfs(anziehen)))
print(sortiere_dfs(studienplan))
print(sortiere_dfs(fehlerhaft))
```

Beim Anziehen kommt hier **dieselbe** Reihenfolge wie bei Kahn heraus, die Probe `ist_gueltig` aus Aufgabe 2.3 bestätigt mit `True`, dass sie gültig ist. Das ist Zufall: Allgemein liefern die beiden Verfahren oft **verschiedene** gültige Reihenfolgen,
weil jedes eine andere Wahl trifft. Beim fehlerhaften Studienplan gibt es keine Reihenfolge, die Funktion antwortet `None`.

</details>

#### Frage 5.2

1. Warum steht ein Knoten in `umgekehrt` **nach** allen seinen Nachfolgern, und warum muss man die Liste am Ende umdrehen?
2. Warum reicht es nicht, nur zwischen „besucht“ und „nicht besucht“ zu unterscheiden? Mit welchem Beispiel (drei Knoten, keine Kreise) würde ein Programm, das jeden Nachbarn im Zustand 2 auch als Kreis meldet, falsch antworten?
3. Beide Verfahren (Kahn und Tiefensuche) brauchen O(K + E). Wo steckt bei der Tiefensuche die Zahl der Kanten, wo die der Knoten?
4. Welches Verfahren würdest du bei sehr langen Ketten (zum Beispiel 100 000 Knoten hintereinander) lieber nehmen, und warum?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `besuche(k)` hängt `k` erst an, wenn die Schleife über alle Nachfolger fertig ist. Alle Nachfolger sind dann schon in der Liste (vor `k`). Wir bekommen also eine Reihenfolge, in der jeder Knoten **hinter** seinen Nachfolgern steht, das ist genau
   falsch herum. Deshalb `umgekehrt.reverse()`.
2. Hat man nur „besucht“ und „nicht besucht“, kann man nicht unterscheiden, ob ein schon besuchter Knoten auf dem **aktuellen Weg** liegt (dann Kreis) oder schon **fertig** ist (dann nicht). Beispiel: `A → B`, `A → C`, `B → C` (ohne Kreis). Von A
   geht es zu B, von B zu C (fertig). Zurück bei A ist C wieder ein schon besuchter Nachfolger, aber **kein** Kreis. Nur Zustand 1 („in Arbeit“) bedeutet einen Kreis.
3. Jeder Knoten wird genau einmal besucht (`zustand[k] == 0` nur einmal), also K Aufrufe von `besuche`. In diesen Aufrufen wird jede Kante einmal angesehen, zusammen E Schritte. Das ergibt O(K + E).
4. Kahn. Die Tiefensuche ruft sich bei einer Kette von 100 000 Knoten 100 000-mal rekursiv auf, und Python bricht bei etwa 1 000 Aufrufen mit einer `RecursionError` ab (man kann die Grenze erhöhen, das ist aber unschön). Kahn braucht keine Rekursion.

</details>

## 6. Wie viele Reihenfolgen gibt es? Und was geht gleichzeitig?

Kahn findet **eine** Reihenfolge. Oft interessiert aber mehr:

- **Wie viele** gültige Reihenfolgen gibt es? Man zählt sie wie beim Backtracking in Notebook 02: Man wählt einen bereiten Knoten, setzt ihn an die nächste Stelle und zählt rekursiv, wie es weitergeht. Danach macht man die Wahl rückgängig und probiert den nächsten bereiten Knoten.
- **Wie schnell** geht es, wenn man mehrere Dinge **gleichzeitig** erledigen kann (zwei Personen beim Anziehen, mehrere Kurse im selben Semester)? Alles, was bereit ist, kommt in dieselbe **Runde** (**Schicht**). Die Zahl der Runden ist die Länge der **längsten Kette** von Vorgaben.

#### Aufgabe 6.1

Die Funktion `zaehle_reihenfolgen(graph)` ist unten vorgegeben. Sie probiert alle Möglichkeiten durch und zählt, wie oft man mit lauter bereiten Knoten bis zum Ende kommt. Ein gesetzter Knoten bekommt zur Markierung den Eingangsgrad `-1`.
Führe sie aus und beantworte danach die Fragen 6.2.

**Erwartete Ausgabe:**
```
6
217
8
0
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def zaehle_reihenfolgen(graph):
    eingang = eingangsgrade(graph)
    return zaehle_weiter(graph, eingang, len(graph))


def zaehle_weiter(graph, eingang, noch_offen):
    if noch_offen == 0:
        return 1                                   # alle Knoten gesetzt: eine vollständige Reihenfolge
    anzahl = 0
    for k in graph:
        if eingang[k] == 0:                        # k ist bereit (und noch nicht gesetzt)
            eingang[k] = -1                        # k ist gesetzt
            for n in graph[k]:
                eingang[n] = eingang[n] - 1
            anzahl = anzahl + zaehle_weiter(graph, eingang, noch_offen - 1)
            for n in graph[k]:                     # Wahl rückgängig machen
                eingang[n] = eingang[n] + 1
            eingang[k] = 0
    return anzahl

klein = baue_graph(['A', 'B', 'C', 'D'], [('A', 'B'), ('C', 'D')])
print(zaehle_reihenfolgen(klein))
print(zaehle_reihenfolgen(anziehen))
print(zaehle_reihenfolgen(studienplan))
print(zaehle_reihenfolgen(fehlerhaft))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def zaehle_reihenfolgen(graph):
    eingang = eingangsgrade(graph)
    return zaehle_weiter(graph, eingang, len(graph))


def zaehle_weiter(graph, eingang, noch_offen):
    if noch_offen == 0:
        return 1                                   # alle Knoten gesetzt: eine vollständige Reihenfolge
    anzahl = 0
    for k in graph:
        if eingang[k] == 0:                        # k ist bereit (und noch nicht gesetzt)
            eingang[k] = -1                        # k ist gesetzt
            for n in graph[k]:
                eingang[n] = eingang[n] - 1
            anzahl = anzahl + zaehle_weiter(graph, eingang, noch_offen - 1)
            for n in graph[k]:                     # Wahl rückgängig machen
                eingang[n] = eingang[n] + 1
            eingang[k] = 0
    return anzahl

klein = baue_graph(['A', 'B', 'C', 'D'], [('A', 'B'), ('C', 'D')])
print(zaehle_reihenfolgen(klein))
print(zaehle_reihenfolgen(anziehen))
print(zaehle_reihenfolgen(studienplan))
print(zaehle_reihenfolgen(fehlerhaft))
```

Das kleine Beispiel `A → B`, `C → D` hat 6 Reihenfolgen: Man mischt die beiden Paare. Beim Anziehen sind es **217**, beim Studienplan 8 (Programmieren muss vor Algorithmen und Datenbanken kommen, Netze darf überall davor stehen, danach folgen Projekt und Abschluss),
beim fehlerhaften Studienplan **0**: Mit einem Kreis gibt es keine gültige Reihenfolge.

</details>

#### Frage 6.2

1. Von den 8! = 40 320 möglichen Anordnungen der Kleidungsstücke sind 217 gültig. Welcher Anteil ist das ungefähr, und was sagt es über die Vorgaben? Wie ändert sich die Zahl, wenn es **weniger** Vorgaben gibt, und wie bei **keiner** Vorgabe?
2. Das Zählen ist nur für kleine Graphen brauchbar, Kahn dagegen für große. Warum? (Denke daran, wie sich die Zahl der Reihenfolgen entwickeln kann.)
3. Der Graph hat die Kanten `A → C`, `B → C`, `C → D`. Wie viele Reihenfolgen gibt es? Rechne zuerst im Kopf, dann prüfe es mit `zaehle_reihenfolgen`.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. 217 von 40 320 sind etwa **0,5 %**. Die Vorgaben schließen also fast alle Anordnungen aus. Mit **weniger** Vorgaben werden es mehr, **ohne** Vorgabe sind alle 8! = 40 320 gültig (jede Anordnung). Jede neue Vorgabe schränkt ein.
2. Die Zahl der Reihenfolgen kann **sehr** groß werden (bei 20 unabhängigen Knoten sind es 20! ≈ 2,4 · 10¹⁸), das Zählen müsste jede einzeln erzeugen. Kahn findet **eine** in O(K + E) Schritten. Wenn man nur eine braucht, zählt man nicht.
3. A und B kommen (in beliebiger Reihenfolge) vor C, dann kommt D: A B C D und B A C D, also **2** Reihenfolgen. (Probe: `print(zaehle_reihenfolgen(baue_graph(['A', 'B', 'C', 'D'], [('A', 'C'), ('B', 'C'), ('C', 'D')])))` gibt 2.)

</details>

#### Aufgabe 6.3

Schreibe `schichten(graph)`: Sie gibt eine Liste von Runden zurück. In Runde 1 stehen alle Knoten mit Eingangsgrad 0. Nach dem Bearbeiten einer ganzen Runde werden bei allen Nachfolgern die Eingangsgrade gesenkt, und wer auf 0 kommt, gehört in die **nächste** Runde.
Gib jede Runde sortiert zurück. Der Aufruf unten zeigt, wie viele Runden man braucht, wenn man alles Mögliche gleichzeitig tut.

**Erwartete Ausgabe:**
```
4 Runden
  Runde 1 : ['Hemd', 'Socken', 'Unterhose']
  Runde 2 : ['Hose', 'Pullover']
  Runde 3 : ['Gürtel', 'Schuhe']
  Runde 4 : ['Jacke']
4 Runden
  Runde 1 : ['Netze', 'Programmieren']
  Runde 2 : ['Algorithmen', 'Datenbanken']
  Runde 3 : ['Projekt']
  Runde 4 : ['Abschluss']
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def schichten(graph):
    eingang = eingangsgrade(graph)
    alle_runden = []
    # dein Code
    return alle_runden

for plan in [anziehen, studienplan]:
    runden = schichten(plan)
    print(len(runden), 'Runden')
    for i in range(len(runden)):
        print('  Runde', i + 1, ':', runden[i])

<details>
<summary>Tipp anzeigen</summary>

Wie bei Kahn, aber statt eines einzelnen Knotens nimmst du die **ganze** Liste `runde` auf einmal. Die Nachfolger, die dabei auf 0 kommen, sammelst du in `naechste`; sie sind die Runde danach.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def ist_gueltig(graph, reihenfolge):
    if sorted(reihenfolge) != sorted(graph):          # jeder Knoten genau einmal?
        return False
    stelle = {}
    for i in range(len(reihenfolge)):
        stelle[reihenfolge[i]] = i
    for k in graph:
        for n in graph[k]:
            if stelle[k] > stelle[n]:                 # Vorgabe k vor n verletzt
                return False
    return True

kurse = ['Programmieren', 'Algorithmen', 'Datenbanken', 'Netze', 'Projekt', 'Abschluss']
voraussetzungen = [('Programmieren', 'Algorithmen'), ('Programmieren', 'Datenbanken'), ('Algorithmen', 'Projekt'),
                   ('Datenbanken', 'Projekt'), ('Netze', 'Projekt'), ('Projekt', 'Abschluss')]
studienplan = baue_graph(kurse, voraussetzungen)
fehlerhaft = baue_graph(kurse, voraussetzungen + [('Projekt', 'Datenbanken')])

kleidung = ['Hemd', 'Hose', 'Gürtel', 'Jacke', 'Pullover', 'Schuhe', 'Socken', 'Unterhose']
vorgaben = [('Unterhose', 'Hose'), ('Hose', 'Schuhe'), ('Socken', 'Schuhe'), ('Hose', 'Gürtel'),
            ('Hemd', 'Pullover'), ('Pullover', 'Jacke'), ('Gürtel', 'Jacke')]

anziehen = baue_graph(kleidung, vorgaben)

def schichten(graph):
    eingang = eingangsgrade(graph)
    runde = []
    for k in graph:
        if eingang[k] == 0:
            runde.append(k)
    alle_runden = []
    while len(runde) > 0:
        alle_runden.append(sorted(runde))
        naechste = []
        for k in runde:
            for n in graph[k]:
                eingang[n] = eingang[n] - 1
                if eingang[n] == 0:
                    naechste.append(n)
        runde = naechste
    return alle_runden

for plan in [anziehen, studienplan]:
    runden = schichten(plan)
    print(len(runden), 'Runden')
    for i in range(len(runden)):
        print('  Runde', i + 1, ':', runden[i])
```

Beim Anziehen reichen **4 Runden**: Hemd, Socken und Unterhose zuerst; dann Hose und Pullover; dann Gürtel und Schuhe; zuletzt die Jacke. Die längste Kette ist Unterhose → Hose → Gürtel → Jacke mit 4 Knoten. Sie bestimmt die
Rundenzahl: Weniger geht nicht, denn die Glieder einer Kette können nie gleichzeitig dran sein. Beim Studienplan sind es 4 Semester: Programmieren und Netze, dann Algorithmen und Datenbanken, dann Projekt, dann Abschluss.

</details>

## Wettbewerbs-Ecke: Schwierigkeiten

*Angelehnt an Aufgabe 2 „Schwierigkeiten“ der 1. Runde des 43. Bundeswettbewerbs Informatik (2024/25).*

Eine Lehrerin hat aus alten Klausuren abgeleitet, welche Aufgaben leichter waren als andere. Jede alte Klausur gibt eine Abstufung wie `B < A < D < F` an (`<` heißt „ist leichter als“). Daraus folgt auch `B < D`, `B < F` und `A < F`.
Für eine neue Klausur wählt sie einige Aufgaben aus. Sie sollen so **angeordnet** werden, dass die leichtere zuerst steht, wenn eine Aufgabe nach den alten Klausuren leichter ist als eine andere. Ein **Konflikt** liegt vor, wenn zwei Aufgaben
nach den Klausuren leichter **und** schwerer als die andere sind: Dann gibt es keine Anordnung.

Wir lösen eine vereinfachte Fassung: Das Programm gibt entweder eine gute Anordnung der gewählten Aufgaben aus, oder, bei einem Konflikt, einen Kreis, der den Widerspruch zeigt. Die Dateien `daten/schwierigkeiten0.txt`, `schwierigkeiten1.txt`, `schwierigkeiten2.txt` und `schwierigkeiten4.txt`
sind Beispiele der Aufgabe. Aufbau: Erste Zeile `n m k` (Zahl der Klausuren, Aufgaben, gewählten Aufgaben), dann `n` Zeilen mit je einer Abstufung, zuletzt eine Zeile mit den `k` gewählten Aufgaben.

**Wichtig:** Eine Beziehung zwischen zwei gewählten Aufgaben kann über eine **nicht gewählte** Aufgabe laufen. Aus `A < E < C` folgt `A < C`, auch wenn `E` nicht gewählt ist. Deshalb baut man zuerst den Graphen über **alle** Aufgaben
und fragt dann, welche Aufgabe von welcher aus **erreichbar** ist.

#### Aufgabe E.1

Schreibe `lies_klausuren(dateiname)`. Sie gibt `(klausuren, gewaehlt)` zurück: `klausuren` ist die Liste der Abstufungen, jede als Liste von Aufgaben (leichteste zuerst), `gewaehlt` die Liste der gewählten Aufgaben.
Tipp: `zeile.split('<')` zerlegt `'B < A < D'` in `['B ', ' A ', ' D']`, `strip()` entfernt die Leerzeichen.

**Erwartete Ausgabe:**
```
[['B', 'A', 'D', 'F'], ['D', 'F', 'G'], ['A', 'E', 'D', 'C'], ['G', 'F', 'C']]
['B', 'C', 'D', 'E', 'F']
```

In [ ]:
def lies_klausuren(dateiname):
    klausuren = []
    gewaehlt = []
    # dein Code
    return klausuren, gewaehlt

klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten0.txt')
print(klausuren)
print(gewaehlt)

<details>
<summary>Tipp anzeigen</summary>

Alle Zeilen mit `datei.read().splitlines()` lesen. In der ersten Zeile steht `n` als erste Zahl (`int(zeilen[0].split()[0])`). Die Zeilen 1 bis `n` sind Klausuren, die Zeile `n + 1` enthält die gewählten Aufgaben (`split()` genügt).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_klausuren(dateiname):
    """Gibt (klausuren, gewaehlt) zurück: klausuren ist eine Liste von Listen (leichteste Aufgabe zuerst)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().splitlines()
    n = int(zeilen[0].split()[0])
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    return klausuren, gewaehlt

klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten0.txt')
print(klausuren)
print(gewaehlt)
```

Das ist das erste Beispiel der Aufgabe: vier Klausuren mit den Aufgaben B bis G und die fünf gewählten Aufgaben B, C, D, E, F (A und G sind nicht gewählt).

</details>

#### Aufgabe E.2

Die Funktionen `baue_klausur_graph(klausuren)` und `erreichbar(graph, start)` sind vorgegeben. Der Graph hat einen Pfeil von jeder Aufgabe zur nächstschwereren **derselben** Klausur. `erreichbar` ist eine Tiefensuche mit einem Stapel
(wie in learn_jwinf3, Notebook 21) und gibt alle Aufgaben zurück, die von `start` aus über Pfeile erreichbar sind, also **schwerer** als `start` sind. Führe die Zelle aus (Beispiel `schwierigkeiten1.txt`) und lies das Ergebnis.

**Erwartete Ausgabe:**
```
A direkt vor ['B', 'E'] | schwerer als A : ['B', 'C', 'D', 'E', 'F', 'G']
B direkt vor ['C'] | schwerer als B : ['C', 'D', 'F']
C direkt vor ['D', 'F'] | schwerer als C : ['D', 'F']
D direkt vor [] | schwerer als D : []
E direkt vor ['C', 'G'] | schwerer als E : ['C', 'D', 'F', 'G']
F direkt vor ['D'] | schwerer als F : ['D']
G direkt vor ['F'] | schwerer als G : ['D', 'F']
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

def lies_klausuren(dateiname):
    """Gibt (klausuren, gewaehlt) zurück: klausuren ist eine Liste von Listen (leichteste Aufgabe zuerst)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().splitlines()
    n = int(zeilen[0].split()[0])
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    return klausuren, gewaehlt

def baue_klausur_graph(klausuren):
    """Graph über alle vorkommenden Aufgaben: Pfeil von jeder Aufgabe zur nächstschwereren derselben Klausur."""
    aufgaben = []
    paare = []
    for klausur in klausuren:
        for i in range(len(klausur)):
            if klausur[i] not in aufgaben:
                aufgaben.append(klausur[i])
            if i + 1 < len(klausur):
                paare.append((klausur[i], klausur[i + 1]))
    return baue_graph(sorted(aufgaben), paare)


def erreichbar(graph, start):
    """Alle Knoten, die von start über Pfeile erreichbar sind (ohne start selbst, außer start liegt in einem Kreis)."""
    gesehen = []
    stapel = [start]
    while len(stapel) > 0:
        k = stapel.pop()
        for n in graph[k]:
            if n not in gesehen:
                gesehen.append(n)
                stapel.append(n)
    return gesehen

klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten1.txt')
graph_s = baue_klausur_graph(klausuren)
for a in graph_s:
    print(a, 'direkt vor', graph_s[a], '| schwerer als', a, ':', sorted(erreichbar(graph_s, a)))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

def lies_klausuren(dateiname):
    """Gibt (klausuren, gewaehlt) zurück: klausuren ist eine Liste von Listen (leichteste Aufgabe zuerst)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().splitlines()
    n = int(zeilen[0].split()[0])
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    return klausuren, gewaehlt

def baue_klausur_graph(klausuren):
    """Graph über alle vorkommenden Aufgaben: Pfeil von jeder Aufgabe zur nächstschwereren derselben Klausur."""
    aufgaben = []
    paare = []
    for klausur in klausuren:
        for i in range(len(klausur)):
            if klausur[i] not in aufgaben:
                aufgaben.append(klausur[i])
            if i + 1 < len(klausur):
                paare.append((klausur[i], klausur[i + 1]))
    return baue_graph(sorted(aufgaben), paare)


def erreichbar(graph, start):
    """Alle Knoten, die von start über Pfeile erreichbar sind (ohne start selbst, außer start liegt in einem Kreis)."""
    gesehen = []
    stapel = [start]
    while len(stapel) > 0:
        k = stapel.pop()
        for n in graph[k]:
            if n not in gesehen:
                gesehen.append(n)
                stapel.append(n)
    return gesehen

klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten1.txt')
graph_s = baue_klausur_graph(klausuren)
for a in graph_s:
    print(a, 'direkt vor', graph_s[a], '| schwerer als', a, ':', sorted(erreichbar(graph_s, a)))
```

Der Graph hat die Aufgaben A bis G. Von `A` aus erreicht man alle anderen sechs Aufgaben, also ist `A` die leichteste. `D` hat keinen Nachfolger und ist die schwerste. Auf dem Weg `A → E → G → F → D`
(Klausuren `A < E < C`, `E < G < F` und `C < F < D`) sieht man: `G` ist leichter als `F` und `D`, obwohl `G` und `D` in keiner Klausur gemeinsam vorkommen. Das Ergebnis von `erreichbar` steht hier **sortiert**, die Reihenfolge der Suche ist beliebig.

</details>

#### Aufgabe E.3

Schreibe `ordne(klausuren, gewaehlt)`:

1. Baue den Graphen über **alle** Aufgaben (`baue_klausur_graph`).
2. Baue daraus einen **zweiten** Graphen nur über die gewählten Aufgaben: Zwischen zwei gewählten Aufgaben `a` und `b` gibt es einen Pfeil `a → b`, wenn `b` von `a` aus **erreichbar** ist (`b != a`).
3. Sortiere ihn mit `sortiere`. Sind alle gewählten Aufgaben dabei, gib die Reihenfolge zurück. Sonst gib `(kreis,)` zurück: ein Tupel mit einem Kreis aus `finde_kreis`.

Der Aufruf unten schreibt für die Beispiele 0, 1, 2 und 4 die Anordnung oder den Konflikt auf.

**Erwartete Ausgabe:**
```
schwierigkeiten0: B C D E F
  Anordnung: B < E < D < F < C
schwierigkeiten1: A C D F G
  Anordnung: A < C < G < F < D
schwierigkeiten2: A B D E F G
  Konflikt, Kreis: D -> E -> D
schwierigkeiten4: B W I N F
  Anordnung: B < I < F < N < W
```

In [ ]:
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

def lies_klausuren(dateiname):
    """Gibt (klausuren, gewaehlt) zurück: klausuren ist eine Liste von Listen (leichteste Aufgabe zuerst)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().splitlines()
    n = int(zeilen[0].split()[0])
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    return klausuren, gewaehlt

def baue_klausur_graph(klausuren):
    """Graph über alle vorkommenden Aufgaben: Pfeil von jeder Aufgabe zur nächstschwereren derselben Klausur."""
    aufgaben = []
    paare = []
    for klausur in klausuren:
        for i in range(len(klausur)):
            if klausur[i] not in aufgaben:
                aufgaben.append(klausur[i])
            if i + 1 < len(klausur):
                paare.append((klausur[i], klausur[i + 1]))
    return baue_graph(sorted(aufgaben), paare)


def erreichbar(graph, start):
    """Alle Knoten, die von start über Pfeile erreichbar sind (ohne start selbst, außer start liegt in einem Kreis)."""
    gesehen = []
    stapel = [start]
    while len(stapel) > 0:
        k = stapel.pop()
        for n in graph[k]:
            if n not in gesehen:
                gesehen.append(n)
                stapel.append(n)
    return gesehen

def ordne(klausuren, gewaehlt):
    # dein Code
    return []

for nr in [0, 1, 2, 4]:
    klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten' + str(nr) + '.txt')
    ergebnis = ordne(klausuren, gewaehlt)
    print('schwierigkeiten' + str(nr) + ':', ' '.join(gewaehlt))
    if isinstance(ergebnis, tuple):
        kreis = ergebnis[0]
        print('  Konflikt, Kreis:', ' -> '.join(kreis + [kreis[0]]))
    else:
        print('  Anordnung:', ' < '.join(ergebnis))

<details>
<summary>Tipp anzeigen</summary>

Für jedes gewählte `a`: `schwerer = erreichbar(graph_alle, a)`; für jedes gewählte `b` mit `b != a` und `b in schwerer` das Paar `(a, b)` sammeln. Dann `graph_gewaehlt = baue_graph(gewaehlt, paare)` und `reihenfolge = sortiere(graph_gewaehlt)`.
Ist die Reihenfolge so lang wie `gewaehlt`, fertig, sonst gibt es einen Kreis. (`finde_kreis` benutzt `uebrige_knoten`, beides steht schon in der Zelle.)

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_graph(knoten, paare):
    graph = {}
    for k in knoten:
        graph[k] = []
    for a, b in paare:
        graph[a].append(b)
    return graph

def eingangsgrade(graph):
    eingang = {}
    for k in graph:
        eingang[k] = 0
    for k in graph:
        for n in graph[k]:
            eingang[n] = eingang[n] + 1
    return eingang

def sortiere(graph):
    eingang = eingangsgrade(graph)
    bereit = []
    for k in graph:
        if eingang[k] == 0:
            bereit.append(k)
    reihenfolge = []
    while len(bereit) > 0:
        k = min(bereit)                       # unter den bereiten Knoten der alphabetisch kleinste
        bereit.remove(k)
        reihenfolge.append(k)
        for n in graph[k]:
            eingang[n] = eingang[n] - 1       # ein wartender Vorgänger weniger
            if eingang[n] == 0:
                bereit.append(n)
    return reihenfolge

def uebrige_knoten(graph):
    fertig = sortiere(graph)
    uebrig = []
    for k in graph:
        if k not in fertig:
            uebrig.append(k)
    return uebrig


def hat_reihenfolge(graph):
    return len(sortiere(graph)) == len(graph)

def finde_kreis(graph):
    """Gibt einen Kreis als Liste zurück, z. B. ['A', 'B', 'C'] für A -> B -> C -> A. Ohne Kreis: leere Liste."""
    uebrig = uebrige_knoten(graph)
    if len(uebrig) == 0:
        return []
    vorgaenger = {}                               # ein Vorgänger, der auch übrig ist
    for k in uebrig:
        for n in graph[k]:
            if n in uebrig:
                vorgaenger[n] = k
    weg = [uebrig[0]]
    while True:
        davor = vorgaenger[weg[-1]]
        if davor in weg:
            kreis = weg[weg.index(davor):]        # ab dem schon bekannten Knoten
            kreis.reverse()                       # wir sind rückwärts gegangen
            return kreis
        weg.append(davor)

def lies_klausuren(dateiname):
    """Gibt (klausuren, gewaehlt) zurück: klausuren ist eine Liste von Listen (leichteste Aufgabe zuerst)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().splitlines()
    n = int(zeilen[0].split()[0])
    klausuren = []
    for i in range(1, n + 1):
        klausur = []
        for teil in zeilen[i].split('<'):
            klausur.append(teil.strip())
        klausuren.append(klausur)
    gewaehlt = zeilen[n + 1].split()
    return klausuren, gewaehlt

def baue_klausur_graph(klausuren):
    """Graph über alle vorkommenden Aufgaben: Pfeil von jeder Aufgabe zur nächstschwereren derselben Klausur."""
    aufgaben = []
    paare = []
    for klausur in klausuren:
        for i in range(len(klausur)):
            if klausur[i] not in aufgaben:
                aufgaben.append(klausur[i])
            if i + 1 < len(klausur):
                paare.append((klausur[i], klausur[i + 1]))
    return baue_graph(sorted(aufgaben), paare)


def erreichbar(graph, start):
    """Alle Knoten, die von start über Pfeile erreichbar sind (ohne start selbst, außer start liegt in einem Kreis)."""
    gesehen = []
    stapel = [start]
    while len(stapel) > 0:
        k = stapel.pop()
        for n in graph[k]:
            if n not in gesehen:
                gesehen.append(n)
                stapel.append(n)
    return gesehen

def ordne(klausuren, gewaehlt):
    """Gibt eine Liste der gewählten Aufgaben in guter Anordnung zurück, bei Konflikt einen Kreis als Tupel (Kreis,)."""
    graph_alle = baue_klausur_graph(klausuren)
    # Graph nur über die gewählten Aufgaben; Pfeil von a nach b, wenn b von a aus erreichbar ist (auch über nicht gewählte)
    paare = []
    for a in gewaehlt:
        schwerer = erreichbar(graph_alle, a)
        for b in gewaehlt:
            if b != a and b in schwerer:
                paare.append((a, b))
    graph_gewaehlt = baue_graph(gewaehlt, paare)
    reihenfolge = sortiere(graph_gewaehlt)
    if len(reihenfolge) == len(gewaehlt):
        return reihenfolge
    return (finde_kreis(graph_gewaehlt),)

for nr in [0, 1, 2, 4]:
    klausuren, gewaehlt = lies_klausuren('daten/schwierigkeiten' + str(nr) + '.txt')
    ergebnis = ordne(klausuren, gewaehlt)
    print('schwierigkeiten' + str(nr) + ':', ' '.join(gewaehlt))
    if isinstance(ergebnis, tuple):
        kreis = ergebnis[0]
        print('  Konflikt, Kreis:', ' -> '.join(kreis + [kreis[0]]))
    else:
        print('  Anordnung:', ' < '.join(ergebnis))
```

Die Ergebnisse stimmen mit dem Lösungsnotebook zur Aufgabe überein: Beispiel 0 `B < E < D < F < C` (die Anordnung `B; E; D; F; C` aus der Aufgabe), Beispiel 1 `A < C < G < F < D` (C und G könnten auch vertauscht sein), Beispiel 4 `B < I < F < N < W`.
Beispiel 2 hat einen **Konflikt**: `D -> E -> D` heißt, dass `E` von `D` aus über Pfeile erreichbar ist und umgekehrt, `D` und `E` sind nach den Klausuren jeweils leichter als die andere. Es gibt keine Anordnung ohne Widerspruch (auch `A` und `B` gehören zu diesem Konflikt).
Bei Beispiel 0 enthält der Graph über alle Aufgaben einen Kreis (`F → G → F`), aber `G` ist **nicht gewählt**, und `F` ist von sich selbst nicht „schwerer“, weil wir `b != a` verlangen. Für die gewählten Aufgaben gibt es also keinen Konflikt.

</details>

#### Frage E.4

1. Warum muss man den Graphen erst über **alle** Aufgaben bauen und kann nicht gleich nur mit den gewählten arbeiten? Gib ein Beispiel aus `schwierigkeiten1.txt`.
2. Beispiel 0 hat einen Kreis `F → G → F` und trotzdem eine gute Anordnung. Beispiel 2 hat einen Kreis und keine. Was ist der Unterschied?
3. Das Programm meldet bei einem Konflikt nur einen Kreis. Die Aufgabe verlangt aber, dass das Programm mit Konflikten **umgeht**. Was könnte man statt der Fehlermeldung tun? (Wie viele Aufgaben stehen bei Beispiel 2 im Kreis?)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Aus `A < E < C` und `E < G < F` folgt `A < G`, obwohl `E` in der Klausur-Auswahl `A C D F G` gar nicht vorkommt. Baute man gleich nur die Pfeile zwischen gewählten Aufgaben, fehlte diese Beziehung. Bei `A C G F D` stünde `G` dann
   vielleicht vor `A`. Die ungewählten Aufgaben liefern also Beziehungen mit.
2. Bei Beispiel 0 liegt `G` im Kreis, aber `G` ist **nicht gewählt**. Von den gewählten Aufgaben kommt nur `F` vor, und eine einzelne Aufgabe widerspricht sich nicht. Bei Beispiel 2 liegen die gewählten Aufgaben A, B, D und E selbst im Kreis. Ein Kreis
   stört also nur, wenn **mindestens zwei gewählte** Aufgaben (über Pfeile) beide leichter und schwerer als die jeweils andere sind.
3. Man könnte Aufgaben im Kreis als **gleich schwer** ansehen: Wer sich gegenseitig erreicht, bildet eine Gruppe, die Gruppen werden wie die Knoten sortiert, innerhalb einer Gruppe ist die Reihenfolge egal. Das macht das Lösungsnotebook
   (Abschnitte „Konflikte, gleich schwere Aufgaben“). Bei Beispiel 2 würden die gewählten Aufgaben A, B, D und E (mit der nicht gewählten Aufgabe C) eine Gruppe bilden.

</details>

## Das hast du gelernt

```python
# Graph mit Nachfolgerlisten und Eingangsgrade
graph = {}
for k in knoten:
    graph[k] = []
for a, b in paare:                         # "a vor b"
    graph[a].append(b)

eingang = {}
for k in graph:
    eingang[k] = 0
for k in graph:
    for n in graph[k]:
        eingang[n] = eingang[n] + 1

# Kahn
bereit = [k for k in graph if eingang[k] == 0]
reihenfolge = []
while len(bereit) > 0:
    k = min(bereit)
    bereit.remove(k)
    reihenfolge.append(k)
    for n in graph[k]:
        eingang[n] = eingang[n] - 1
        if eingang[n] == 0:
            bereit.append(n)
if len(reihenfolge) < len(graph):
    print('Kreis, keine Reihenfolge!')
```

- Eine **topologische Sortierung** ordnet die Knoten eines gerichteten Graphen so an, dass jede Kante von vorn nach hinten zeigt. Es gibt eine, wenn der Graph **keinen Kreis** hat.
- **Kahn:** Eingangsgrade zählen, Knoten ohne offene Vorgänger bearbeiten, bei ihren Nachfolgern den Eingangsgrad senken. Bleiben Knoten übrig, gibt es einen **Kreis**. Rückwärts über übrige Vorgänger findet man ihn.
- **Tiefensuche:** Knoten nach allen Nachfolgern anhängen und die Liste umdrehen; Zustand „in Arbeit“ zeigt einen Kreis.
- Immer die **Länge** der Reihenfolge prüfen (sonst gilt eine unvollständige Liste als Ergebnis) und eine **Probe** (`ist_gueltig`) schreiben.
- Die Zahl der Reihenfolgen zählt man mit Backtracking, die **Schichten** (alles Bereite gleichzeitig) zeigen die längste Kette.
- Laufzeit **O(K + E)**: jeder Knoten und jede Kante einmal.
- Beziehungen können über nicht betroffene Knoten laufen: erst Graph über alles, dann Erreichbarkeit.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Schwierigkeiten](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/43_Schwierigkeiten/schwierigkeiten_loesung.ipynb) (43. BwInf): Klausuraufgaben anordnen, Kreise als gleich schwere Aufgaben behandeln

Im nächsten Notebook, **09 Bäume**, geht es um Strukturen, die ineinander verschachtelt sind, zum Beispiel Klammerausdrücke. Ein **Baum** ist ein Graph ohne Kreis, in dem jeder Knoten (außer der Wurzel) genau einen Vorgänger hat, und man wertet ihn rekursiv aus.